# Movie Agent

In [7]:
import json
import asyncio
import requests
from dotenv import load_dotenv
from agents import Agent, Runner, SQLiteSession, function_tool

load_dotenv()

MODEL = "gpt-4o-mini"
MOVIE_API_BASE_URL = "https://nomad-movies.nomadcoders.workers.dev"
EXIT_COMMANDS = {"q", "quit", "exit"}

session = SQLiteSession(
    session_id="default_user",
    db_path="chat_history.db"
)

In [8]:
def fetch_json(path: str):
    response = requests.get(f"{MOVIE_API_BASE_URL}{path}", timeout=20)
    response.raise_for_status()
    return response.json()


@function_tool
def get_popular_movies():
    """Get current popular movies."""
    movies = fetch_json("/movies")
    return json.dumps(
        [
            {
                "id": movie["id"],
                "title": movie["title"],
                "overview": movie.get("overview", ""),
                "release_date": movie.get("release_date"),
                "vote_average": movie.get("vote_average"),
            }
            for movie in movies[:10]
        ],
        ensure_ascii=False,
    )


@function_tool
def get_movie_details(id: int):
    """Get detailed information for a movie by id."""
    movie = fetch_json(f"/movies/{id}")
    return json.dumps(
        {
            "id": movie["id"],
            "title": movie["title"],
            "overview": movie.get("overview", ""),
            "genres": [genre["name"] for genre in movie.get("genres", [])],
            "runtime": movie.get("runtime"),
            "release_date": movie.get("release_date"),
            "vote_average": movie.get("vote_average"),
        },
        ensure_ascii=False,
    )


@function_tool
def get_movie_credits(id: int):
    """Get cast and crew information for a movie by id."""
    credits = fetch_json(f"/movies/{id}/credits")
    cast = [
        {
            "name": person["name"],
            "character": person.get("character"),
        }
        for person in credits
        if person.get("known_for_department") == "Acting"
    ][:10]
    crew = [
        {
            "name": person["name"],
            "job": person.get("known_for_department"),
        }
        for person in credits
        if person.get("known_for_department") != "Acting"
    ][:10]

    return json.dumps(
        {
            "cast": cast,
            "crew": crew,
        },
        ensure_ascii=False,
    )

In [9]:
agent = Agent(
    name="Movie Assistant",
    model=MODEL,
    instructions=(
        "You are a movie recommendation assistant. "
        "Use the movie API tools when the user asks about popular movies, "
        "movie details, cast, or crew. Keep answers concise and useful."
    ),
    tools=[get_popular_movies, get_movie_details, get_movie_credits]
)

In [10]:
async def main():
    print("Movie Agent를 시작합니다. 종료하려면 q, quit, exit 를 입력하세요.")

    while True:
        try:
            user_input = input("대화를 입력하세요: ").strip()
        except EOFError:
            break

        if user_input.lower() in EXIT_COMMANDS:
            print("대화를 종료합니다.")
            break

        if not user_input:
            continue

        print(f"User: {user_input}")

        result = await Runner.run(agent, user_input, session=session)
        print(f"AI: {result.final_output}")

In [11]:
await main()

Movie Agent를 시작합니다. 종료하려면 q, quit, exit 를 입력하세요.
User: 안녕 이 전에 내가 어떤 영화를 추천 받았지?
AI: 이전에 추천해드린 영화는 **"Avatar: Fire and Ash"**입니다. 이 영화는 제이크 설리와 네이티리가 판도라에서 새로운 위협에 맞서 싸우는 내용을 담고 있습니다. 더 궁금한 점이 있으시면 말씀해 주세요!
User: 더 자세히 알려주고 비슷한 장르 영화 3개만 추천해줘
AI: **영화: Avatar: Fire and Ash**

- **개요**: RDA와의 전쟁 이후, 제이크 설리와 그의 아내 네이티리는 무자비한 나비족인 '재야족(Ash People)'의 공격에 직면합니다. 이들 그룹은 강력한 리더 바랑(Varang)에 의해 이끌리며, 제이크의 가족은 그들의 생존과 판도라의 미래를 위해 싸워야 합니다.
- **장르**: 공상 과학, 모험, 판타지
- **러닝 타임**: 198분
- **개봉일**: 2025-12-17
- **평점**: 7.4

**비슷한 장르 영화 추천**:

1. **Dune (2021)**
   - 먼 미래의 사막 행성 아라키스에서의 권력과 생존을 위한 전투 이야기.
   
2. **Guardians of the Galaxy (2014)**
   - 다양한 우주 종족들이 협력하여 악과 맞서 싸우는 유머와 모험이 가득한 이야기.
   
3. **Star Wars: The Force Awakens (2015)**
   - 새로운 세대의 영웅들이 제국의 잔당과 싸우며 전설의 여정을 이어가는 이야기.

이 영화들은 모두 판타지와 모험의 요소가 가득하니 즐기실 수 있을 거예요! 더 필요한 정보가 있다면 말씀해 주세요.
대화를 종료합니다.
